# Terminal × product roles (demand / supply centers)

Classify each **Installation × Product** (and optional **complex × Product**) from
`trades_imp_exp_master` over **full history**. Algeciras FO was the discovery
example; the pipeline covers **all products** the same way.

**Why a separate notebook from `03`:** role math + a monthly cube are heavier.
Keep `03` for light terminal profiles; use this for product-level typology.

**Defaults**
- Status: `Delivered`
- Grain: Product (not Family)
- Window: full history
- Complexes: manual map in `data/derived/terminal_complexes.csv`

**Units**
- `kb` = `Volume (bbl) / 1000` (absolute; import and export both positive)
- `kbd` = `kb / days_in_month` on the month grain (same idea as the ES/FR/PT tracker)
- Lifetime tables expose average monthly `kbd` by direction

**PyGWalker tip:** your FO chart used monthly X on the event master. Here we walk a
**monthly aggregate** of that master (same chart grain, ~450k rows vs ~1.5M events)
with `kernel_computation=True`. Signed measures (`kb_signed`, `kbd_signed`) keep
imports below zero like `kb_imp_exp` on the raw file.

In [26]:
from __future__ import annotations

import calendar
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import pygwalker as pyg

ROOT = Path("..").resolve()
TABLEAU_DIR = ROOT / "data" / "derived" / "tableau"
DERIVED_DIR = ROOT / "data" / "derived"
PARQUET_PATH = TABLEAU_DIR / "trades_imp_exp_master.parquet"
CSV_PATH = TABLEAU_DIR / "trades_imp_exp_master.csv"
COMPLEXES_PATH = DERIVED_DIR / "terminal_complexes.csv"

OUT_MONTHLY = DERIVED_DIR / "terminal_product_monthly.parquet"
OUT_ROLES = DERIVED_DIR / "terminal_product_roles.parquet"
OUT_ROLES_CSV = DERIVED_DIR / "terminal_product_roles.csv"
OUT_COMPLEX_ROLES = DERIVED_DIR / "complex_product_roles.parquet"
OUT_PATTERNS = DERIVED_DIR / "terminal_product_patterns.parquet"
OUT_PATTERNS_CSV = DERIVED_DIR / "terminal_product_patterns.csv"

STATUS = "Delivered"

# Role thresholds (same spirit as the Algeciras visual discovery)
IMPORT_HEAVY_MIN = 0.70
EXPORT_HEAVY_MAX = 0.30
MIN_TOTAL_KB = 500.0  # lifetime floor so thin flows are not labeled as centers

# Briefing charts / tables (FO was only the discovery example — cover all products)
FOCUS_COUNTRY = "Spain"
TOP_N_PER_PRODUCT = 5  # ranked demand/supply centers shown per product
EXAMPLE_INSTALLATIONS = [
    "Vopak Algeciras",
    "CLH Algeciras Storage",
    "Gibraltar-San Roque Refinery",
]

# Pattern monitor: last-3m softness + seasonality context
RECENT_MONTHS = 3
MIN_PATTERN_MONTHS = 24  # need enough history for YoY / z-scores
SOFT_Z = -1.0  # recent 3m vs all prior months
SOFT_SEASONAL_RATIO = 0.70  # recent 3m vs same calendar months in prior years
MIN_SOFT_MEAN_KBD = 5.0  # ignore tiny series when raising soft flags
ROLLING_ROLE_MONTHS = 12

ROLE_COLORS = {
    "demand_center": "#c45c26",
    "supply_center": "#2f6f8f",
    "balanced_hub": "#6b7280",
    "thin": "#d1d5db",
}

print("pygwalker", getattr(pyg, "__version__", "?"))
print("ROOT", ROOT)

pygwalker 0.5.0.1
ROOT C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\prod_nowcast


## 1) Load delivered events

Same source you used in PyGWalker (`trades_imp_exp_master`). We keep only columns
needed for monthly product flows.

In [27]:
USE_COLS = [
    "Imp/Exp",
    "kb_imp_exp",
    "Installation",
    "Country",
    "Zone",
    "Continent",
    "Subcontinent",
    "TradingRegion",
    "Date",
    "Product",
    "Volume (bbl)",
    "Trade status",
]

if PARQUET_PATH.exists():
    raw = pd.read_parquet(PARQUET_PATH, columns=USE_COLS)
    source = PARQUET_PATH.name
else:
    assert CSV_PATH.exists(), f"Missing master under {TABLEAU_DIR}"
    raw = pd.read_csv(CSV_PATH, usecols=USE_COLS, parse_dates=["Date"])
    source = CSV_PATH.name

events = raw.loc[raw["Trade status"].eq(STATUS)].copy()
events = events.loc[
    events["Installation"].notna()
    & events["Installation"].ne("")
    & events["Product"].notna()
    & events["Product"].ne("")
    & events["Date"].notna()
]

# Absolute kb for comparable import/export; keep signed kb for FO-style charts.
events["kb_abs"] = events["Volume (bbl)"] / 1000.0
events["year_month"] = events["Date"].dt.to_period("M").dt.to_timestamp()

print(f"{source}: {len(raw):,} → {len(events):,} usable {STATUS} rows")
print("range:", events["Date"].min(), "→", events["Date"].max())

trades_imp_exp_master.parquet: 1,545,966 → 1,406,959 usable Delivered rows
range: 2016-01-02 19:09:00 → 2026-07-16 09:59:00


In [28]:
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()  # notebook under notebooks/
# ROOT = Path(r"C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\prod_nowcast")  # if standalone
PARQUET_PATH = ROOT / "data" / "derived" / "tableau" / "trades_imp_exp_master.parquet"
OUT_PATH = ROOT / "notebooks" / "spain_terminals_unique.csv"

COLS = [
    "Installation",
    "Country",
    "Zone",
    "Continent",
    "Subcontinent",
    "TradingRegion",
]

def mode_or_first(s: pd.Series):
    s = s.dropna()
    s = s[s.astype(str).str.strip().ne("")]
    if s.empty:
        return pd.NA
    m = s.mode()
    return m.iloc[0] if len(m) else s.iloc[0]

# Reuse `events` if already loaded in the notebook; otherwise load Delivered rows.
if "events" not in globals():
    raw = pd.read_parquet(PARQUET_PATH, columns=COLS + ["Trade status", "Date"])
    events = raw.loc[
        raw["Trade status"].eq("Delivered")
        & raw["Installation"].notna()
        & raw["Installation"].ne("")
    ].copy()

spain = events.loc[events["Country"].eq("Spain"), COLS].copy()

spain_terminals = (
    spain.groupby("Installation", dropna=False)
    .agg(
        Country=("Country", mode_or_first),
        Zone=("Zone", mode_or_first),
        Continent=("Continent", mode_or_first),
        Subcontinent=("Subcontinent", mode_or_first),
        TradingRegion=("TradingRegion", mode_or_first),
    )
    .reset_index()
    .rename(columns={"Installation": "installation_name"})
    .sort_values("installation_name")
    .reset_index(drop=True)
)

#spain_terminals.to_csv(OUT_PATH, index=False)
print(f"{len(spain_terminals)} unique Spain terminals → {OUT_PATH}")
spain_terminals.head()

69 unique Spain terminals → C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\prod_nowcast\notebooks\spain_terminals_unique.csv


,installation_name,Country,Zone,Continent,Subcontinent,TradingRegion
0,APM Terminals Barcelona,Spain,Barcelona,Europe,Southern Europe,MED Zone (MED Sea+Black Sea)
1,ASESA,Spain,Tarragona,Europe,Southern Europe,MED Zone (MED Sea+Black Sea)
2,Alcoa San Ciprian,Spain,San Cibrao,Europe,Southern Europe,North West Europe Zone
3,Alpenor,Spain,Vigo,Europe,Southern Europe,North West Europe Zone
4,Arrecife,Spain,Arrecife,Europe,Southern Europe,Other Southern Europe


## 2) Monthly cube (chart grain)

Long format close to the master: one row per
`Installation × Product × month × Imp/Exp`.

- `kbd = kb_abs / days_in_that_month`
- `kbd_signed` / `kb_signed` mirror master sign (+ export, − import)

In [29]:
def days_in_timestamp_month(ts: pd.Timestamp) -> int:
    return calendar.monthrange(ts.year, ts.month)[1]


def mode_or_first(s: pd.Series):
    m = s.dropna()
    if m.empty:
        return pd.NA
    modes = m.mode()
    return modes.iloc[0] if len(modes) else m.iloc[0]


monthly = (
    events.groupby(
        [
            "Installation",
            "Country",
            "Zone",
            "Continent",
            "Subcontinent",
            "TradingRegion",
            "Product",
            "year_month",
            "Imp/Exp",
        ],
        dropna=False,
        observed=True,
    )
    .agg(
        kb_abs=("kb_abs", "sum"),
        kb_signed=("kb_imp_exp", "sum"),
        n_trades=("kb_abs", "size"),
    )
    .reset_index()
)

monthly["days_in_month"] = monthly["year_month"].map(days_in_timestamp_month)
monthly["kbd"] = monthly["kb_abs"] / monthly["days_in_month"]
monthly["kbd_signed"] = monthly["kb_signed"] / monthly["days_in_month"]

print(f"monthly rows: {len(monthly):,}")
monthly.head(3)

monthly rows: 447,573


,Installation,Country,Zone,Continent,Subcontinent,TradingRegion,Product,year_month,Imp/Exp,kb_abs,kb_signed,n_trades,days_in_month,kbd,kbd_signed
0,A & E Tank Farm,Nigeria,Warri,Africa,Western Africa,Western Africa,Blending Comps,2021-01-01,Import,155.031,-155.031,1,31,5.001000,-5.001000
1,A & E Tank Farm,Nigeria,Warri,Africa,Western Africa,Western Africa,Blending Comps,2024-06-01,Import,162.119,-162.119,1,30,5.403967,-5.403967
2,A & E Tank Farm,Nigeria,Warri,Africa,Western Africa,Western Africa,Diesel,2024-05-01,Import,238.735,-238.735,2,31,7.701129,-7.701129


## 3) Lifetime Installation × Product roles

| role | rule |
|---|---|
| `demand_center` | `import_share ≥ 0.70` and `total_kb ≥ MIN_TOTAL_KB` |
| `supply_center` | `import_share ≤ 0.30` and `total_kb ≥ MIN_TOTAL_KB` |
| `balanced_hub` | in between and above the volume floor |
| `thin` | below `MIN_TOTAL_KB` |

`import_kbd_avg` / `export_kbd_avg` = mean monthly `kbd` for that direction
(averaged over months where that Installation×Product×direction had volume).

In [30]:
def classify_role(import_share: float, total_kb: float) -> str:
    if total_kb < MIN_TOTAL_KB or pd.isna(import_share):
        return "thin"
    if import_share >= IMPORT_HEAVY_MIN:
        return "demand_center"
    if import_share <= EXPORT_HEAVY_MAX:
        return "supply_center"
    return "balanced_hub"


# Stable geo attrs once per installation (avoids splitting roles on rare geo drift).
geo = events.groupby("Installation", dropna=False).agg(
    Country=("Country", mode_or_first),
    Zone=("Zone", mode_or_first),
    Continent=("Continent", mode_or_first),
    Subcontinent=("Subcontinent", mode_or_first),
    TradingRegion=("TradingRegion", mode_or_first),
)

dir_stats = (
    monthly.groupby(["Installation", "Product", "Imp/Exp"], dropna=False)
    .agg(
        kb=("kb_abs", "sum"),
        kbd_avg=("kbd", "mean"),
        n_months=("year_month", "nunique"),
        first_month=("year_month", "min"),
        last_month=("year_month", "max"),
    )
    .reset_index()
)

wide = dir_stats.pivot_table(
    index=["Installation", "Product"],
    columns="Imp/Exp",
    values=["kb", "kbd_avg", "n_months", "first_month", "last_month"],
    aggfunc="first",
)
wide.columns = [f"{metric}_{direction.lower()}" for metric, direction in wide.columns]
roles = wide.reset_index()

for c, default in [
    ("kb_import", 0.0),
    ("kb_export", 0.0),
    ("kbd_avg_import", 0.0),
    ("kbd_avg_export", 0.0),
    ("n_months_import", 0),
    ("n_months_export", 0),
]:
    if c not in roles.columns:
        roles[c] = default
    else:
        roles[c] = roles[c].fillna(default)

roles = roles.rename(
    columns={
        "kb_import": "import_kb",
        "kb_export": "export_kb",
        "kbd_avg_import": "import_kbd_avg",
        "kbd_avg_export": "export_kbd_avg",
    }
)

roles["total_kb"] = roles["import_kb"] + roles["export_kb"]
roles["net_kb"] = roles["import_kb"] - roles["export_kb"]
roles["import_share"] = roles["import_kb"] / roles["total_kb"].replace(0, pd.NA)
roles["total_kbd_avg"] = roles["import_kbd_avg"].fillna(0) + roles["export_kbd_avg"].fillna(0)
roles["n_months"] = roles[["n_months_import", "n_months_export"]].max(axis=1)
roles["first_month"] = roles[[c for c in roles.columns if c.startswith("first_month_")]].min(axis=1)
roles["last_month"] = roles[[c for c in roles.columns if c.startswith("last_month_")]].max(axis=1)
roles["role"] = [
    classify_role(s, t) for s, t in zip(roles["import_share"], roles["total_kb"])
]

term_total = roles.groupby("Installation")["total_kb"].transform("sum")
roles["product_share_of_terminal"] = roles["total_kb"] / term_total.replace(0, pd.NA)

roles = roles.merge(geo, left_on="Installation", right_index=True, how="left")

print(roles["role"].value_counts())
print(f"role rows: {len(roles):,}")

role
thin             8728
demand_center    5399
supply_center    2369
balanced_hub     1218
Name: count, dtype: int64
role rows: 17,714


## 4) Complex map + complex × product roles

Manual CSV so storage sites can be joined without merging the adjacent refinery.
Starter: **Algeciras Storage** = Vopak Algeciras + CLH Algeciras Storage
(San Roque left unmapped on purpose).

In [31]:
complexes = pd.read_csv(COMPLEXES_PATH)
complexes["installation"] = complexes["installation"].astype(str).str.strip()
cmap = complexes.rename(columns={"installation": "Installation"})[
    ["Installation", "complex_id", "complex_name"]
]

roles = roles.merge(cmap, on="Installation", how="left")
monthly = monthly.merge(cmap, on="Installation", how="left")

# Sum member volumes inside each complex-month, then average kbd across months.
complex_monthly = (
    monthly.loc[monthly["complex_id"].notna()]
    .groupby(
        ["complex_id", "complex_name", "Product", "year_month", "Imp/Exp"],
        dropna=False,
    )
    .agg(kb_abs=("kb_abs", "sum"), kbd=("kbd", "sum"))
    .reset_index()
)

c_dir = (
    complex_monthly.groupby(["complex_id", "complex_name", "Product", "Imp/Exp"], dropna=False)
    .agg(
        kb=("kb_abs", "sum"),
        kbd_avg=("kbd", "mean"),
        n_months=("year_month", "nunique"),
        first_month=("year_month", "min"),
        last_month=("year_month", "max"),
    )
    .reset_index()
)

c_wide = c_dir.pivot_table(
    index=["complex_id", "complex_name", "Product"],
    columns="Imp/Exp",
    values=["kb", "kbd_avg", "n_months", "first_month", "last_month"],
    aggfunc="first",
)
c_wide.columns = [f"{metric}_{direction.lower()}" for metric, direction in c_wide.columns]
complex_roles = c_wide.reset_index()

for c, default in [
    ("kb_import", 0.0),
    ("kb_export", 0.0),
    ("kbd_avg_import", 0.0),
    ("kbd_avg_export", 0.0),
]:
    if c not in complex_roles.columns:
        complex_roles[c] = default
    else:
        complex_roles[c] = complex_roles[c].fillna(default)

complex_roles = complex_roles.rename(
    columns={
        "kb_import": "import_kb",
        "kb_export": "export_kb",
        "kbd_avg_import": "import_kbd_avg",
        "kbd_avg_export": "export_kbd_avg",
    }
)
complex_roles["total_kb"] = complex_roles["import_kb"] + complex_roles["export_kb"]
complex_roles["net_kb"] = complex_roles["import_kb"] - complex_roles["export_kb"]
complex_roles["import_share"] = complex_roles["import_kb"] / complex_roles[
    "total_kb"
].replace(0, pd.NA)
complex_roles["total_kbd_avg"] = (
    complex_roles["import_kbd_avg"].fillna(0) + complex_roles["export_kbd_avg"].fillna(0)
)
complex_roles["role"] = [
    classify_role(s, t)
    for s, t in zip(complex_roles["import_share"], complex_roles["total_kb"])
]
complex_roles["n_installations"] = complex_roles["complex_id"].map(
    complexes.groupby("complex_id")["installation"].nunique()
)

print("installations mapped into a complex:", roles["complex_id"].notna().groupby(roles["Installation"]).any().sum())
print("complex × product roles:", len(complex_roles))
complex_roles.sort_values("total_kb", ascending=False).head(8)

installations mapped into a complex: 4
complex × product roles: 28


,complex_id,complex_name,Product,first_month_export,first_month_import,export_kb,import_kb,export_kbd_avg,import_kbd_avg,last_month_export,last_month_import,n_months_export,n_months_import,total_kb,net_kb,import_share,total_kbd_avg,role,n_installations
7,algeciras_storage,Algeciras Storage,FO,2017-01-01,2016-01-01,32864.367,195131.522,10.103137,55.397183,2026-06-01,2026-07-01,107.0,116.0,227995.889,162267.155,0.855855,65.500320,demand_center,2
23,bilbao_port,Bilbao Ref+Port,Gasoline,2017-01-01,2017-01-01,81805.339,3521.957,23.789540,3.044596,2026-06-01,2026-03-01,113.0,38.0,85327.296,-78283.382,0.041276,26.834136,supply_center,2
22,bilbao_port,Bilbao Ref+Port,Gasoil,2017-01-01,2017-02-01,53748.274,26454.955,16.489354,11.868267,2026-06-01,2026-05-01,107.0,73.0,80203.229,-27293.319,0.329849,28.357622,balanced_hub,2
19,bilbao_port,Bilbao Ref+Port,Diesel,2017-03-01,2017-01-01,10684.825,63997.435,6.860998,22.865085,2026-06-01,2026-06-01,51.0,92.0,74682.260,53312.610,0.856930,29.726083,demand_center,2
26,bilbao_port,Bilbao Ref+Port,Naphtha,2017-01-01,2017-04-01,46295.356,4203.853,13.565562,3.831494,2026-07-01,2026-03-01,112.0,36.0,50499.209,-42091.503,0.083246,17.397056,supply_center,2
6,algeciras_storage,Algeciras Storage,Dirty Feedstocks,2017-09-01,2017-01-01,1696.087,29097.430,3.088238,11.933394,2025-10-01,2026-03-01,18.0,80.0,30793.517,27401.343,0.944921,15.021632,demand_center,2
8,algeciras_storage,Algeciras Storage,Gasoil,2017-03-01,2017-06-01,4270.163,25601.241,4.657530,8.765698,2026-06-01,2026-06-01,30.0,96.0,29871.404,21331.078,0.857048,13.423228,demand_center,2
21,bilbao_port,Bilbao Ref+Port,FO,2017-01-01,2017-04-01,21067.288,7749.492,10.033788,10.229052,2026-01-01,2025-05-01,69.0,25.0,28816.780,-13317.796,0.268923,20.262840,supply_center,2


## 5) Understanding the output (all products)

FO at Algeciras was the **discovery example**. The systematic object is every
`Installation × Product` row in `roles` (and mapped rows in `complex_roles`).

| Table | Grain | Use it to… |
|---|---|---|
| `monthly` | install × product × month × Imp/Exp | recreate time charts (`kbd` / `kbd_signed`) |
| `roles` | install × product (full history) | label demand / supply / hub / thin |
| `complex_roles` | complex × product | same labels after desk-merging sites |

**How to read a role row**
- `import_share` high → inbound-heavy for that product (`demand_center` if also large enough)
- `import_kbd_avg` / `export_kbd_avg` → typical monthly flow rate in that direction
- `product_share_of_terminal` → how important that product is inside the site’s mix
- `thin` → below `MIN_TOTAL_KB`; ignore for typology

Briefing charts below use `FOCUS_COUNTRY` (default Spain) and **all products**.

In [32]:
focus = roles.loc[roles["Country"].eq(FOCUS_COUNTRY)].copy()
focus_labeled = focus.loc[focus["role"].ne("thin")].copy()

print(f"{FOCUS_COUNTRY}: {focus['Installation'].nunique()} terminals | {len(focus):,} install×product rows")
print(f"non-thin rows used in charts: {len(focus_labeled):,}")
print("role mix (rows):")
print(focus["role"].value_counts())
print("\nrole mix by product (rows):")
display(
    focus.pivot_table(index="Product", columns="role", values="Installation", aggfunc="count", fill_value=0)
    .reindex(columns=["demand_center", "supply_center", "balanced_hub", "thin"], fill_value=0)
    .sort_index()
)

# Top demand / supply centers per product (systematic scan across the book)
ranked_parts = []
for role_name in ("demand_center", "supply_center"):
    # demand: high import_kbd; supply: high export_kbd
    metric = "import_kbd_avg" if role_name == "demand_center" else "export_kbd_avg"
    chunk = (
        focus.loc[focus["role"].eq(role_name)]
        .sort_values(["Product", metric], ascending=[True, False])
        .groupby("Product", group_keys=False)
        .head(TOP_N_PER_PRODUCT)
        .assign(rank_within_product=lambda d: d.groupby("Product").cumcount() + 1)
    )
    ranked_parts.append(chunk)

ranked = pd.concat(ranked_parts, ignore_index=True)
print(f"\nTop {TOP_N_PER_PRODUCT} {FOCUS_COUNTRY} centers per product (demand + supply):")
display(
    ranked[
        [
            "role",
            "Product",
            "rank_within_product",
            "Installation",
            "Zone",
            "complex_name",
            "import_share",
            "import_kbd_avg",
            "export_kbd_avg",
            "total_kb",
            "product_share_of_terminal",
        ]
    ].sort_values(["role", "Product", "rank_within_product"])
)

# Example sites from the discovery path — all products, not FO-only
example = focus.loc[focus["Installation"].isin(EXAMPLE_INSTALLATIONS)].sort_values(
    ["Installation", "total_kb"], ascending=[True, False]
)
print("\nExample installations (discovery path) — product roles:")
display(
    example[
        [
            "Installation",
            "Product",
            "role",
            "import_share",
            "import_kbd_avg",
            "export_kbd_avg",
            "total_kb",
            "product_share_of_terminal",
            "complex_name",
        ]
    ]
)

if len(complex_roles):
    print("\nMapped complexes — all products:")
    display(
        complex_roles.sort_values(["complex_name", "total_kb"], ascending=[True, False])[
            [
                "complex_name",
                "Product",
                "role",
                "import_share",
                "import_kbd_avg",
                "export_kbd_avg",
                "total_kb",
                "n_installations",
            ]
        ]
    )

Spain: 69 terminals | 515 install×product rows
non-thin rows used in charts: 266
role mix (rows):
role
thin             249
demand_center    159
balanced_hub      54
supply_center     53
Name: count, dtype: int64

role mix by product (rows):


role,demand_center,supply_center,balanced_hub,thin
Product,,,,
Bitumen/Asphalt,4,5,1,22
Blending Comps,14,4,7,24
CBFS,0,2,1,15
Clean Condensate,2,1,0,10
Coal Tar,2,0,0,12
Cutter Stock,1,0,0,6
Diesel,24,5,5,10
Dirty Feedstocks,13,2,5,11
FO,19,6,3,14



Top 5 Spain centers per product (demand + supply):


,role,Product,rank_within_product,Installation,Zone,complex_name,import_share,import_kbd_avg,export_kbd_avg,total_kb,product_share_of_terminal
0,demand_center,Bitumen/Asphalt,1,Gijon,Gijon,NaN,1.0,1.199305,0.000000,3367.769,0.080057
1,demand_center,Bitumen/Asphalt,2,Tepsa Valencia,Valencia,NaN,0.965486,1.019084,0.518408,2746.407,0.053826
2,demand_center,Bitumen/Asphalt,3,Tenerife Refinery,Tenerife,NaN,0.957191,0.893614,1.136533,796.474,0.005612
3,demand_center,Bitumen/Asphalt,4,Asfaltos Arinaga,Arinaga,NaN,0.77048,0.870685,0.855626,2170.018,1.000000
4,demand_center,Blending Comps,1,Barcelona 32F & 32G,Barcelona,NaN,0.949241,5.055186,1.912409,6977.731,0.084594
...,...,...,...,...,...,...,...,...,...,...,...
100,supply_center,Naphtha,1,Repsol Cartagena Refinery,Cartagena,NaN,0.022735,3.128960,41.580541,147529.384,0.313097
101,supply_center,Naphtha,2,Torre Arenillas,Huelva,NaN,0.025754,1.388391,21.706308,73968.054,0.344880
102,supply_center,Naphtha,3,Bilbao Oil Facilities,Bilbao,Bilbao Ref+Port,0.083611,3.831494,13.501995,50278.506,0.150012
103,supply_center,Naphtha,4,Repsol Coruna Refinery,Coruna,NaN,0.013266,3.252762,11.787570,37497.265,0.179190



Example installations (discovery path) — product roles:


,Installation,Product,role,import_share,import_kbd_avg,export_kbd_avg,total_kb,product_share_of_terminal,complex_name
2157,CLH Algeciras Storage,FO,demand_center,0.948111,14.676918,1.714545,53099.541,0.788399,Algeciras Storage
2158,CLH Algeciras Storage,Gasoil,demand_center,0.994146,3.758107,1.506800,7722.118,0.114655,Algeciras Storage
2156,CLH Algeciras Storage,Dirty Feedstocks,demand_center,0.754446,4.494929,2.276804,3124.246,0.046387,Algeciras Storage
2155,CLH Algeciras Storage,Diesel,demand_center,1.0,3.630117,0.000000,883.392,0.013116,Algeciras Storage
2162,CLH Algeciras Storage,SRFO,demand_center,0.901059,2.823086,1.101274,676.898,0.010050,Algeciras Storage
2159,CLH Algeciras Storage,Gasoline,demand_center,0.883555,3.789140,0.985435,524.686,0.007790,Algeciras Storage
2161,CLH Algeciras Storage,Naphtha,thin,0.958347,3.055627,0.661226,492.119,0.007307,Algeciras Storage
2151,CLH Algeciras Storage,Bitumen/Asphalt,thin,0.715138,2.973067,1.201569,257.233,0.003819,Algeciras Storage
2152,CLH Algeciras Storage,Blending Comps,thin,1.0,1.946524,0.000000,240.652,0.003573,Algeciras Storage
2160,CLH Algeciras Storage,Jet,thin,0.867521,1.256525,0.572548,133.976,0.001989,Algeciras Storage



Mapped complexes — all products:


,complex_name,Product,role,import_share,import_kbd_avg,export_kbd_avg,total_kb,n_installations
7,Algeciras Storage,FO,demand_center,0.855855,55.397183,10.103137,227995.889,2
6,Algeciras Storage,Dirty Feedstocks,demand_center,0.944921,11.933394,3.088238,30793.517,2
8,Algeciras Storage,Gasoil,demand_center,0.857048,8.765698,4.657530,29871.404,2
13,Algeciras Storage,SRFO,demand_center,0.987222,6.400345,1.186301,8532.163,2
5,Algeciras Storage,Diesel,demand_center,1.000000,4.723467,0.000000,3015.430,2
9,Algeciras Storage,Gasoline,demand_center,0.777846,3.911959,2.131656,2891.437,2
14,Algeciras Storage,Slurry,demand_center,0.966982,3.516088,1.292645,1213.658,2
12,Algeciras Storage,Naphtha,demand_center,0.982916,3.538947,0.661226,1199.852,2
4,Algeciras Storage,Cutter Stock,demand_center,0.971779,1.866472,0.604200,642.279,2
2,Algeciras Storage,CBFS,balanced_hub,0.328118,1.581797,1.407527,573.599,2


### Charts — country briefing (all products)

1. **Role mix by product** — where demand vs supply labels concentrate  
2. **Import vs export kbd scatter** — each point is one Installation × Product  
3. **Top centers per product** — demand (import kbd) and supply (export kbd)  
4. **Example installations heatmap** — `import_share` across products (Algeciras path)

In [33]:
# --- 1) Role mix by product (volume-weighted, clearer than row counts) ---
mix = (
    focus.groupby(["Product", "role"], dropna=False)["total_kb"]
    .sum()
    .reset_index()
)
fig_mix = px.bar(
    mix,
    x="Product",
    y="total_kb",
    color="role",
    color_discrete_map=ROLE_COLORS,
    category_orders={"role": ["demand_center", "supply_center", "balanced_hub", "thin"]},
    title=f"{FOCUS_COUNTRY}: lifetime kb by product and role",
    labels={"total_kb": "total kb", "Product": "product"},
)
fig_mix.update_layout(xaxis_tickangle=-35, legend_title_text="role", height=420)
fig_mix.show()

# --- 2) Demand / supply map: every non-thin Installation × Product ---
fig_scatter = px.scatter(
    focus_labeled,
    x="export_kbd_avg",
    y="import_kbd_avg",
    color="role",
    size="total_kb",
    size_max=36,
    hover_name="Installation",
    hover_data={
        "Product": True,
        "import_share": ":.2f",
        "Zone": True,
        "complex_name": True,
        "total_kb": ":.0f",
        "export_kbd_avg": ":.2f",
        "import_kbd_avg": ":.2f",
        "role": True,
    },
    color_discrete_map=ROLE_COLORS,
    title=f"{FOCUS_COUNTRY}: import vs export kbd (point = Installation × Product)",
    labels={
        "export_kbd_avg": "export kbd (avg month)",
        "import_kbd_avg": "import kbd (avg month)",
    },
)
fig_scatter.update_layout(height=520)
fig_scatter.show()

# --- 3) Where demand / supply centers sit, by product (treemap) ---
demand = focus.loc[focus["role"].eq("demand_center")].copy()
supply = focus.loc[focus["role"].eq("supply_center")].copy()

if len(demand):
    fig_demand = px.treemap(
        demand,
        path=["Product", "Installation"],
        values="import_kbd_avg",
        color="import_share",
        color_continuous_scale="Oranges",
        title=f"{FOCUS_COUNTRY}: demand centers (size = import kbd avg, color = import share)",
        hover_data={"total_kb": ":.0f", "Zone": True},
    )
    fig_demand.update_layout(height=560, margin=dict(t=50, l=10, r=10, b=10))
    fig_demand.show()
else:
    print("No demand_center rows in focus country.")

if len(supply):
    fig_supply = px.treemap(
        supply,
        path=["Product", "Installation"],
        values="export_kbd_avg",
        color="import_share",
        color_continuous_scale="Blues_r",
        title=f"{FOCUS_COUNTRY}: supply centers (size = export kbd avg, color = import share)",
        hover_data={"total_kb": ":.0f", "Zone": True},
    )
    fig_supply.update_layout(height=560, margin=dict(t=50, l=10, r=10, b=10))
    fig_supply.show()
else:
    print("No supply_center rows in focus country.")

# --- 4) Example installations: import_share heatmap across products ---
ex = focus.loc[focus["Installation"].isin(EXAMPLE_INSTALLATIONS)].copy()
if ex.empty:
    print("EXAMPLE_INSTALLATIONS not found in focus country — skip heatmap.")
else:
    heat = ex.pivot_table(
        index="Installation",
        columns="Product",
        values="import_share",
        aggfunc="first",
    )
    # Order products by total kb at these sites so the matrix is readable
    prod_order = (
        ex.groupby("Product")["total_kb"].sum().sort_values(ascending=False).index.tolist()
    )
    heat = heat.reindex(columns=prod_order)

    fig_heat = px.imshow(
        heat,
        color_continuous_scale="RdYlBu_r",
        zmin=0,
        zmax=1,
        aspect="auto",
        title=(
            "Example sites — import_share by product "
            f"(≥{IMPORT_HEAVY_MIN:.0%} ≈ demand, ≤{EXPORT_HEAVY_MAX:.0%} ≈ supply)"
        ),
        labels={"color": "import_share"},
    )
    fig_heat.update_layout(height=320 + 40 * len(heat), xaxis_tickangle=-35)
    fig_heat.show()

    # Tiny annotation: role tags under the same matrix idea
    role_mat = ex.pivot_table(
        index="Installation", columns="Product", values="role", aggfunc="first"
    ).reindex(columns=prod_order)
    print("Role tags for the same example matrix:")
    display(role_mat)

Role tags for the same example matrix:


Product,FO,Gasoil,Gasoline,Jet,Dirty Feedstocks,Naphtha,Kerosene,SRFO,Diesel,Bitumen/Asphalt,Blending Comps,Slurry,Cutter Stock,CBFS,Clean Condensate,Coal Tar
Installation,,,,,,,,,,,,,,,,
CLH Algeciras Storage,demand_center,demand_center,demand_center,thin,demand_center,thin,NaN,demand_center,demand_center,thin,thin,thin,thin,thin,NaN,NaN
Gibraltar-San Roque Refinery,supply_center,supply_center,supply_center,demand_center,balanced_hub,supply_center,demand_center,demand_center,supply_center,supply_center,balanced_hub,thin,thin,thin,supply_center,thin
Vopak Algeciras,demand_center,demand_center,demand_center,NaN,demand_center,demand_center,thin,demand_center,demand_center,thin,thin,demand_center,demand_center,balanced_hub,NaN,thin


In [34]:
# Example sites from the discovery path — all products, not FO-only
installation_search = ['Repsol Petronor Refinery','Bilbao Oil Facilities']

example = focus.loc[focus["Installation"].isin(installation_search)].sort_values(
    ["Installation", "total_kb"], ascending=[True, False]
)
print("\nExample installations (discovery path) — product roles:")
display(
    example[
        [
            "Installation",
            "Product",
            "role",
            "import_share",
            "import_kbd_avg",
            "export_kbd_avg",
            "total_kb",
            "product_share_of_terminal",
            "complex_name",
        ]
    ]
)


Example installations (discovery path) — product roles:


,Installation,Product,role,import_share,import_kbd_avg,export_kbd_avg,total_kb,product_share_of_terminal,complex_name
1659,Bilbao Oil Facilities,Gasoline,supply_center,0.041057,2.961960,23.326770,83640.732,0.249553,Bilbao Ref+Port
1658,Bilbao Oil Facilities,Gasoil,balanced_hub,0.327103,11.774469,16.332461,79114.952,0.236050,Bilbao Ref+Port
1655,Bilbao Oil Facilities,Diesel,demand_center,0.853918,22.284256,6.860998,73142.523,0.218230,Bilbao Ref+Port
1662,Bilbao Oil Facilities,Naphtha,supply_center,0.083611,3.831494,13.501995,50278.506,0.150012,Bilbao Ref+Port
1657,Bilbao Oil Facilities,FO,supply_center,0.152506,6.875654,8.738071,20686.655,0.061721,Bilbao Ref+Port
1656,Bilbao Oil Facilities,Dirty Feedstocks,supply_center,0.165018,6.219591,7.665279,15872.975,0.047359,Bilbao Ref+Port
1652,Bilbao Oil Facilities,Blending Comps,demand_center,0.842234,2.241539,1.625759,5519.369,0.016468,Bilbao Ref+Port
1660,Bilbao Oil Facilities,Jet,demand_center,0.77406,9.275958,6.219452,2560.017,0.007638,Bilbao Ref+Port
1663,Bilbao Oil Facilities,SRFO,balanced_hub,0.522275,7.486933,6.908120,2186.446,0.006524,Bilbao Ref+Port
1653,Bilbao Oil Facilities,CBFS,supply_center,0.0,0.000000,1.644681,994.604,0.002968,Bilbao Ref+Port


## 6) Pattern monitor — predictability & soft demand/supply

Lifetime `roles` answer “what kind of center is this?”. This section asks:

1. **Is the Imp/Exp pattern historically regular?** (CV, weak/strong YoY seasonality, rolling role stability)
2. **Is the last 3 months soft vs history?** for import (demand proxy) and export (supply proxy)

Soft flags use **both**:
- `z` vs all prior months (level shock)
- `seasonal_ratio` = recent 3m mean / mean of the **same calendar months** in prior years  
  (so a quiet winter is not flagged just because summers are busy)

Requires `MIN_PATTERN_MONTHS` of history. Missing months inside a series span are treated as 0 kbd (no cargo that month).

In [35]:
def _complete_monthly_kbd(year_month: pd.Series, values: pd.Series) -> pd.Series:
    """Month-start index from first to last activity; gaps -> 0 kbd."""
    tmp = pd.DataFrame({"year_month": year_month, "kbd": values}).sort_values("year_month")
    tmp = tmp.groupby("year_month", as_index=False)["kbd"].sum()
    idx = pd.date_range(tmp["year_month"].min(), tmp["year_month"].max(), freq="MS")
    s = tmp.set_index("year_month")["kbd"].reindex(idx, fill_value=0.0)
    s.index.name = "year_month"
    return s


def _direction_softness(s: pd.Series) -> dict:
    """Last RECENT_MONTHS vs prior history + same-calendar-month baseline."""
    if len(s) < MIN_PATTERN_MONTHS:
        return {
            "recent_kbd": np.nan,
            "hist_mean": np.nan,
            "z": np.nan,
            "pctile": np.nan,
            "seasonal_expected": np.nan,
            "seasonal_ratio": np.nan,
            "soft": False,
        }

    recent = s.iloc[-RECENT_MONTHS:]
    hist = s.iloc[:-RECENT_MONTHS]
    recent_kbd = float(recent.mean())
    hist_mean = float(hist.mean())
    hist_std = float(hist.std(ddof=0))
    z = (recent_kbd - hist_mean) / hist_std if hist_std > 1e-9 else np.nan
    pctile = float((hist <= recent_kbd).mean()) if len(hist) else np.nan

    # Same calendar months in prior years (seasonality-aware baseline)
    seasonal_vals = []
    for ts, val in recent.items():
        prior_same = hist[hist.index.month == ts.month]
        if len(prior_same):
            seasonal_vals.append(float(prior_same.mean()))
    seasonal_expected = float(np.mean(seasonal_vals)) if seasonal_vals else np.nan
    seasonal_ratio = (
        recent_kbd / seasonal_expected
        if pd.notna(seasonal_expected) and seasonal_expected > 1e-9
        else np.nan
    )

    soft = False
    if recent_kbd >= 0 and hist_mean >= MIN_SOFT_MEAN_KBD:
        z_soft = pd.notna(z) and z <= SOFT_Z
        seas_soft = pd.notna(seasonal_ratio) and seasonal_ratio <= SOFT_SEASONAL_RATIO
        soft = bool(z_soft or seas_soft)

    return {
        "recent_kbd": recent_kbd,
        "hist_mean": hist_mean,
        "z": z,
        "pctile": pctile,
        "seasonal_expected": seasonal_expected,
        "seasonal_ratio": seasonal_ratio,
        "soft": soft,
    }


def _yoy_month_corr(s: pd.Series) -> float:
    """Mean corr of each calendar month with the prior year (seasonal predictability)."""
    corrs = []
    df = s.rename("v").reset_index()
    df["month"] = df["year_month"].dt.month
    for _, gm in df.groupby("month"):
        gm = gm.sort_values("year_month")
        if len(gm) < 3:
            continue
        with np.errstate(invalid="ignore", divide="ignore"):
            c = gm["v"].corr(gm["v"].shift(1))
        if pd.notna(c):
            corrs.append(float(c))
    return float(np.mean(corrs)) if corrs else np.nan


def pattern_stats_for_group(g: pd.DataFrame) -> pd.Series:
    """One Installation×Product (or complex×Product) monthly long slice -> pattern row."""
    # Aggregate Imp/Exp within month if duplicate geo rows slipped in
    wide = (
        g.groupby(["year_month", "Imp/Exp"], dropna=False)["kbd"]
        .sum()
        .unstack(fill_value=0.0)
        .reset_index()
    )
    for col in ("Import", "Export"):
        if col not in wide.columns:
            wide[col] = 0.0
    imp = _complete_monthly_kbd(wide["year_month"], wide["Import"])
    exp = _complete_monthly_kbd(wide["year_month"], wide["Export"]).reindex(
        imp.index, fill_value=0.0
    )
    total = imp + exp

    empty = {
        "n_months": int(len(imp)),
        "enough_history": False,
        "import_cv": np.nan,
        "export_cv": np.nan,
        "import_yoy_corr": np.nan,
        "export_yoy_corr": np.nan,
        "import_recent_kbd": np.nan,
        "export_recent_kbd": np.nan,
        "import_hist_mean_kbd": np.nan,
        "export_hist_mean_kbd": np.nan,
        "import_z": np.nan,
        "export_z": np.nan,
        "import_pctile": np.nan,
        "export_pctile": np.nan,
        "import_seasonal_expected": np.nan,
        "export_seasonal_expected": np.nan,
        "import_seasonal_ratio": np.nan,
        "export_seasonal_ratio": np.nan,
        "import_soft": False,
        "export_soft": False,
        "roll_import_share_mean": np.nan,
        "roll_import_share_std": np.nan,
        "role_stability": np.nan,
        "asof_month": imp.index.max() if len(imp) else pd.NaT,
    }
    if len(imp) < MIN_PATTERN_MONTHS:
        return pd.Series(empty)

    imp_soft = _direction_softness(imp)
    exp_soft = _direction_softness(exp)

    share = (imp / total.replace(0, np.nan)).fillna(0.5)
    roll = share.rolling(ROLLING_ROLE_MONTHS, min_periods=ROLLING_ROLE_MONTHS)
    roll_mean = float(roll.mean().iloc[-1]) if roll.mean().notna().any() else np.nan
    roll_std = float(roll.std(ddof=0).iloc[-1]) if roll.std(ddof=0).notna().any() else np.nan
    # Stability: share of last ROLLING months that agree with lifetime-style bucket of roll_mean
    recent_share = share.iloc[-ROLLING_ROLE_MONTHS:]
    if pd.isna(roll_mean):
        role_stability = np.nan
    elif roll_mean >= IMPORT_HEAVY_MIN:
        role_stability = float((recent_share >= IMPORT_HEAVY_MIN).mean())
    elif roll_mean <= EXPORT_HEAVY_MAX:
        role_stability = float((recent_share <= EXPORT_HEAVY_MAX).mean())
    else:
        role_stability = float(
            ((recent_share > EXPORT_HEAVY_MAX) & (recent_share < IMPORT_HEAVY_MIN)).mean()
        )

    def _cv(x: pd.Series) -> float:
        mu = float(x.mean())
        return float(x.std(ddof=0) / mu) if mu > 1e-9 else np.nan

    return pd.Series(
        {
            "n_months": int(len(imp)),
            "enough_history": True,
            "import_cv": _cv(imp),
            "export_cv": _cv(exp),
            "import_yoy_corr": _yoy_month_corr(imp),
            "export_yoy_corr": _yoy_month_corr(exp),
            "import_recent_kbd": imp_soft["recent_kbd"],
            "export_recent_kbd": exp_soft["recent_kbd"],
            "import_hist_mean_kbd": imp_soft["hist_mean"],
            "export_hist_mean_kbd": exp_soft["hist_mean"],
            "import_z": imp_soft["z"],
            "export_z": exp_soft["z"],
            "import_pctile": imp_soft["pctile"],
            "export_pctile": exp_soft["pctile"],
            "import_seasonal_expected": imp_soft["seasonal_expected"],
            "export_seasonal_expected": exp_soft["seasonal_expected"],
            "import_seasonal_ratio": imp_soft["seasonal_ratio"],
            "export_seasonal_ratio": exp_soft["seasonal_ratio"],
            "import_soft": imp_soft["soft"],
            "export_soft": exp_soft["soft"],
            "roll_import_share_mean": roll_mean,
            "roll_import_share_std": roll_std,
            "role_stability": role_stability,
            "asof_month": imp.index.max(),
        }
    )


# Build on focus-country monthly rows (faster); patterns are still install×product.
# Explicit loop avoids groupby.apply column-alignment quirks across short/long series.
monthly_focus = monthly.loc[monthly["Country"].eq(FOCUS_COUNTRY)].copy()
pattern_rows = []
for (installation, product), g in monthly_focus.groupby(
    ["Installation", "Product"], dropna=False
):
    row = pattern_stats_for_group(g)
    row["Installation"] = installation
    row["Product"] = product
    pattern_rows.append(row)
patterns = pd.DataFrame(pattern_rows)

# Attach geo / lifetime role for reading
patterns = patterns.merge(
    roles[
        [
            "Installation",
            "Product",
            "Country",
            "Zone",
            "complex_name",
            "role",
            "import_share",
            "import_kbd_avg",
            "export_kbd_avg",
            "total_kb",
        ]
    ],
    on=["Installation", "Product"],
    how="left",
)

print(
    f"{FOCUS_COUNTRY} pattern rows: {len(patterns):,} | "
    f"enough history: {int(patterns['enough_history'].sum()):,}"
)
print(
    "soft flags — import:",
    int(patterns["import_soft"].sum()),
    "| export:",
    int(patterns["export_soft"].sum()),
)
print("as-of month:", patterns["asof_month"].max())
print("\nimport YoY corr (enough history):")
print(patterns.loc[patterns["enough_history"], "import_yoy_corr"].describe())

soft_import = patterns.loc[
    patterns["import_soft"] & patterns["enough_history"]
].sort_values("import_z")
soft_export = patterns.loc[
    patterns["export_soft"] & patterns["enough_history"]
].sort_values("export_z")

print(f"\nSoft IMPORT (demand proxy) — last {RECENT_MONTHS}m:")
display(
    soft_import[
        [
            "Installation",
            "Product",
            "role",
            "import_recent_kbd",
            "import_hist_mean_kbd",
            "import_z",
            "import_seasonal_ratio",
            "import_yoy_corr",
            "role_stability",
            "complex_name",
        ]
    ].head(25)
)

print(f"\nSoft EXPORT (supply proxy) — last {RECENT_MONTHS}m:")
display(
    soft_export[
        [
            "Installation",
            "Product",
            "role",
            "export_recent_kbd",
            "export_hist_mean_kbd",
            "export_z",
            "export_seasonal_ratio",
            "export_yoy_corr",
            "role_stability",
            "complex_name",
        ]
    ].head(25)
)

print("\nExample installations — pattern snapshot:")
display(
    patterns.loc[patterns["Installation"].isin(EXAMPLE_INSTALLATIONS)]
    .sort_values(["Installation", "total_kb"], ascending=[True, False])[
        [
            "Installation",
            "Product",
            "role",
            "import_yoy_corr",
            "import_cv",
            "import_z",
            "import_seasonal_ratio",
            "import_soft",
            "export_z",
            "export_seasonal_ratio",
            "export_soft",
            "role_stability",
        ]
    ]
)

Spain pattern rows: 515 | enough history: 370
soft flags — import: 9 | export: 10
as-of month: 2026-07-01 00:00:00

import YoY corr (enough history):
count    308.000000
mean      -0.128278
std        0.190053
min       -1.000000
25%       -0.217466
50%       -0.134386
75%       -0.030334
max        0.484652
Name: import_yoy_corr, dtype: float64

Soft IMPORT (demand proxy) — last 3m:


,Installation,Product,role,import_recent_kbd,import_hist_mean_kbd,import_z,import_seasonal_ratio,import_yoy_corr,role_stability,complex_name
278,Gibraltar-San Roque Refinery,Kerosene,demand_center,1.329378,5.644137,-0.846739,0.195087,-0.024795,0.500000,NaN
136,CEPSA Las Palmas,FO,demand_center,9.127475,15.969838,-0.734359,0.491259,-0.060472,0.916667,NaN
445,Tenerife Refinery,Jet,demand_center,4.529294,9.084354,-0.648715,0.635611,-0.049977,0.916667,NaN
506,Vopak Terquimsa Barcelona,Gasoline,balanced_hub,2.502072,6.773632,-0.559777,0.391015,-0.101383,0.750000,NaN
443,Tenerife Refinery,Gasoil,demand_center,10.124356,14.901752,-0.537396,0.698931,0.130627,1.000000,NaN
505,Vopak Terquimsa Barcelona,Gasoil,demand_center,5.127398,11.074614,-0.459417,0.458849,-0.097398,0.666667,NaN
337,Palma de Mallorca,Gasoil,demand_center,3.198647,5.569632,-0.455828,0.595033,-0.002728,0.833333,NaN
442,Tenerife Refinery,FO,demand_center,3.377661,5.425821,-0.425373,0.605646,-0.095604,0.666667,NaN
326,Oryx Las Palmas,Gasoil,balanced_hub,12.226928,16.878140,-0.388243,0.669662,0.049078,0.416667,NaN



Soft EXPORT (supply proxy) — last 3m:


,Installation,Product,role,export_recent_kbd,export_hist_mean_kbd,export_z,export_seasonal_ratio,export_yoy_corr,role_stability,complex_name
477,Torre Arenillas,Naphtha,supply_center,8.821704,20.888593,-1.133001,0.414093,-0.030473,0.916667,NaN
361,Repsol Cartagena Refinery,Diesel,supply_center,8.104963,20.831292,-0.856735,0.388770,-0.025044,0.750000,NaN
51,BP Castellon Refinery,Gasoline,supply_center,9.980370,18.141554,-0.853030,0.478951,-0.031212,1.000000,NaN
274,Gibraltar-San Roque Refinery,FO,supply_center,0.438731,5.814869,-0.810103,0.070681,-0.186176,0.583333,NaN
491,Vopak Algeciras,FO,demand_center,1.897740,7.939287,-0.793071,0.273591,-0.207751,0.833333,Algeciras Storage
476,Torre Arenillas,Kerosene,supply_center,2.015905,6.830280,-0.779284,0.237436,-0.090561,0.750000,NaN
431,Tarragona Refinery,Gasoil,supply_center,1.540323,5.493161,-0.537124,0.273204,-0.048251,0.416667,NaN
473,Torre Arenillas,Gasoil,supply_center,12.266916,17.401770,-0.458633,0.678671,0.089001,1.000000,NaN
365,Repsol Cartagena Refinery,Gasoline,supply_center,4.034257,5.963803,-0.382285,0.579569,-0.037132,0.166667,NaN
297,La Rabida,FO,supply_center,3.089908,5.347286,-0.354514,0.451452,-0.142541,0.333333,NaN



Example installations — pattern snapshot:


,Installation,Product,role,import_yoy_corr,import_cv,import_z,import_seasonal_ratio,import_soft,export_z,export_seasonal_ratio,export_soft,role_stability
152,CLH Algeciras Storage,FO,demand_center,-0.034165,0.588734,-0.290744,0.867925,False,-0.641515,0.000000,False,1.000000
153,CLH Algeciras Storage,Gasoil,demand_center,0.079670,1.145915,0.185201,1.234096,False,-0.099504,NaN,False,0.750000
151,CLH Algeciras Storage,Dirty Feedstocks,demand_center,-0.215958,2.950593,-0.256376,0.290367,False,-0.266161,0.000000,False,0.750000
150,CLH Algeciras Storage,Diesel,demand_center,-0.195131,4.604624,0.155860,2.500525,False,NaN,NaN,False,0.916667
157,CLH Algeciras Storage,SRFO,demand_center,-0.239052,3.502437,2.231805,12.551951,False,-0.162660,NaN,False,0.833333
154,CLH Algeciras Storage,Gasoline,demand_center,-0.250000,5.907433,0.107249,0.513910,False,-0.177632,NaN,False,0.916667
156,CLH Algeciras Storage,Naphtha,thin,-0.198053,5.997316,0.337324,18.626312,False,-0.106000,NaN,False,0.833333
146,CLH Algeciras Storage,Bitumen/Asphalt,thin,-0.250000,8.341997,0.082434,NaN,False,-0.163277,NaN,False,0.916667
147,CLH Algeciras Storage,Blending Comps,thin,-0.166667,5.824673,0.353768,1.439581,False,NaN,NaN,False,0.916667
155,CLH Algeciras Storage,Jet,thin,-0.250000,6.533135,-0.156344,0.000000,False,NaN,NaN,False,0.833333


### Pattern charts

1. **Predictability scatter** — YoY corr vs CV (lumpy vs seasonal)  
2. **Soft map** — recent import z vs seasonal ratio (demand softness)  
3. **Example site timelines** — monthly import/export kbd for `EXAMPLE_INSTALLATIONS`

In [36]:
pat_ok = patterns.loc[patterns["enough_history"]].copy()
pat_ok["label"] = pat_ok["Installation"] + " · " + pat_ok["Product"]

# 1) Lumpy vs seasonal
fig_pred = px.scatter(
    pat_ok,
    x="import_cv",
    y="import_yoy_corr",
    color="role",
    size="total_kb",
    size_max=32,
    hover_name="label",
    color_discrete_map=ROLE_COLORS,
    title=f"{FOCUS_COUNTRY}: import predictability (YoY corr vs CV)",
    labels={
        "import_cv": "import CV (higher = lumpier)",
        "import_yoy_corr": "import YoY month corr (higher = more seasonal)",
    },
)
fig_pred.add_hline(y=0, line_dash="dot", line_color="#999")
fig_pred.update_layout(height=480)
fig_pred.show()

# 2) Soft demand monitor
soft_plot = pat_ok.loc[pat_ok["import_hist_mean_kbd"] >= MIN_SOFT_MEAN_KBD].copy()
soft_plot["soft_flag"] = soft_plot["import_soft"].map({True: "soft", False: "normal"})
fig_soft = px.scatter(
    soft_plot,
    x="import_seasonal_ratio",
    y="import_z",
    color="soft_flag",
    color_discrete_map={"soft": "#c45c26", "normal": "#9ca3af"},
    hover_name="label",
    hover_data={
        "role": True,
        "import_recent_kbd": ":.2f",
        "import_hist_mean_kbd": ":.2f",
        "import_seasonal_ratio": ":.2f",
        "import_z": ":.2f",
    },
    title=(
        f"{FOCUS_COUNTRY}: last-{RECENT_MONTHS}m import softness "
        f"(z≤{SOFT_Z} or seasonal_ratio≤{SOFT_SEASONAL_RATIO})"
    ),
    labels={
        "import_seasonal_ratio": "recent / same-calendar-month history",
        "import_z": "recent z vs all prior months",
    },
)
fig_soft.add_hline(y=SOFT_Z, line_dash="dash", line_color="#c45c26")
fig_soft.add_vline(x=SOFT_SEASONAL_RATIO, line_dash="dash", line_color="#c45c26")
fig_soft.update_layout(height=480)
fig_soft.show()

# 3) Example timelines (all products at discovery sites)
ex_m = monthly_focus.loc[
    monthly_focus["Installation"].isin(EXAMPLE_INSTALLATIONS)
].copy()
if ex_m.empty:
    print("No EXAMPLE_INSTALLATIONS rows in monthly_focus.")
else:
    ex_wide = (
        ex_m.groupby(
            ["Installation", "Product", "year_month", "Imp/Exp"], dropna=False
        )["kbd"]
        .sum()
        .reset_index()
    )
    fig_ex = px.line(
        ex_wide,
        x="year_month",
        y="kbd",
        color="Product",
        line_dash="Imp/Exp",
        facet_row="Installation",
        title=f"Example sites — monthly kbd by product (Imp/Exp)",
        height=280 * ex_wide["Installation"].nunique(),
    )
    fig_ex.update_yaxes(matches=None, title_text="kbd")
    fig_ex.update_layout(legend_title_text="")
    fig_ex.show()

## 7) Persist derived tables

Re-run after changing thresholds, complex map, pattern knobs, or the master extract.

In [37]:
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

monthly.to_parquet(OUT_MONTHLY, index=False)
roles.to_parquet(OUT_ROLES, index=False)
roles.to_csv(OUT_ROLES_CSV, index=False)
complex_roles.to_parquet(OUT_COMPLEX_ROLES, index=False)
patterns.to_parquet(OUT_PATTERNS, index=False)
patterns.to_csv(OUT_PATTERNS_CSV, index=False)

print("wrote", OUT_MONTHLY.name, f"({len(monthly):,} rows)")
print("wrote", OUT_ROLES.name, f"({len(roles):,} rows)")
print("wrote", OUT_ROLES_CSV.name)
print("wrote", OUT_COMPLEX_ROLES.name, f"({len(complex_roles):,} rows)")
print("wrote", OUT_PATTERNS.name, f"({len(patterns):,} rows)")
print("wrote", OUT_PATTERNS_CSV.name)

wrote terminal_product_monthly.parquet (447,573 rows)
wrote terminal_product_roles.parquet (17,714 rows)
wrote terminal_product_roles.csv
wrote complex_product_roles.parquet (28 rows)
wrote terminal_product_patterns.parquet (515 rows)
wrote terminal_product_patterns.csv


## 8) PyGWalker — monthly explorer (recommended)

Same grain as your discovery chart, for **any** product / site:

1. Filter `Country` = Spain (or pick installations / a `complex_name`)
2. X = `year_month`
3. Y = `Imp/Exp` + `sum(kbd_signed)` (or `sum(kb_signed)`)
4. Color = `Product`  
   Optional: also filter `Product` once you care about one grade

In [38]:
walker_monthly = pyg.walk(
    monthly,
    kernel_computation=True,
    default_tab="vis",
)
walker_monthly

Box(children=(HTML(value='\n<div id="ifr-pyg-0006571e070e6c10TO92phzVCnUYbged" style="height: auto">\n    <hea…

## 9) PyGWalker — lifetime roles

Good for typology across the product book:

1. Filter `Country` = Spain
2. Filter `role` ∈ demand_center / supply_center (drop `thin`)
3. X = `Installation`, Y = `import_kbd_avg` or `export_kbd_avg`, Color = `Product`  
   Or scatter: X = `export_kbd_avg`, Y = `import_kbd_avg`, Color = `role`, Size = `total_kb`

In [39]:
walker_roles = pyg.walk(
    roles,
    kernel_computation=True,
    default_tab="vis",
)
walker_roles

Box(children=(HTML(value='\n<div id="ifr-pyg-0006571e07116832fCm5ouNdqULzeBgO" style="height: auto">\n    <hea…